In [ ]:
device = "0"
model_name = "meta-llama/Llama-3.2-3B"
experiment = "corruption"
corruption_type = "substitution"
dataset_size = 200
max_length = 50
lam = 256
evaluate_scores = True

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

In [ ]:
if corruption_type not in ["substitution", "deletion", "insertion"]:
    raise ValueError(f"Unknown corruption type: {corruption_type}")

In [ ]:
import torch
from utils import load_model_and_tokenizer, load_c4

experiment = "varying_bits"

device = torch.device(f"cuda:{device}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model, tokenizer = load_model_and_tokenizer(model_name, device)

data = load_c4(tokenizer, dataset_size)
data[0]

/home/ubuntu/anaconda3/envs/wepa/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda:0


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.17s/it]


Model: meta-llama/Llama-3.2-3B
Loading dataset from cache...


{'input_ids': tensor([[128000,    791,  59796,   6406,   8925,   6787,    311,   7055,   7884,
          13658,    389,   7231,   8339,    400,   1419,   3610,    323,   1023,
          36580,    311,   1977,    264,  26097,  15679,    304,  29016,   4409,
             11,    719,   1193,   1306,  11011,  12483,    315,  18671,  13286,
          11062,    323,  28424,  49262,    369,    477,   2403,    279,   2447,
            627,    791,   4330,  44650,   4580]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}

In [ ]:
from tqdm import tqdm
import wandb

from utils import calc_roc_auc_score, calc_tpr_at_fpr, init_wandb, UnbiasedWatermarker, get_wat_name


if corruption_type in ('substitution', 'deletion'):
    corruption_fractions = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
elif corruption_type == 'insertion':
    corruption_fractions = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
else:
    raise ValueError(f"Unknown corruption type: {corruption_type}")

vocab_size = model.config.vocab_size


def random_substitution(token_ids, fraction):
    n_tokens = int(fraction * len(token_ids))
    indices = torch.randperm(len(token_ids))[:n_tokens]

    # Randomly substitute a fraction of the output sequence
    token_ids = torch.tensor(token_ids)
    token_ids[indices] = torch.randint(0, vocab_size, (n_tokens,), device=device)

    return token_ids


def random_deletion(token_ids, fraction):
    n_tokens = int(fraction * len(token_ids))
    indices = torch.randperm(len(token_ids), device=device)[:n_tokens]
    mask = torch.ones(len(token_ids), dtype=torch.bool, device=device)
    mask[indices] = 0
    # Randomly delete a fraction of the output sequence
    token_ids = torch.masked_select(torch.tensor(token_ids), mask)

    return token_ids


def random_insertion(token_ids, fraction):
    n_tokens = int(fraction * len(token_ids))
    random_tokens = torch.randint(0, vocab_size, (n_tokens,))

    # Insert tokens at random positions
    token_ids = torch.tensor(token_ids)
    for i in range(n_tokens):
        index = torch.randint(0, len(token_ids) + 1, (1,), device=device)
        token_ids = torch.cat(
            [
                token_ids[:index],
                torch.tensor([random_tokens[i]], device=device),
                token_ids[index:],
            ]
        )

    return token_ids


if corruption_type == "substitution":
    corruption_func = random_substitution
elif corruption_type == "deletion":
    corruption_func = random_deletion
elif corruption_type == "insertion":
    corruption_func = random_insertion
else:
    raise ValueError(f"Unknown corruption type: {corruption_type}")


def run_experiment(wat, corruption_fraction):
    print(f"Running experiment: {wat}")

    wat_name = get_wat_name(wat)

    p_values = []
    all_token_ids = []
    for i in tqdm(range(dataset_size)):
        for j in range(10):
            inputs = data[i].to(device)
            outputs = wat.generate(
                model,
                inputs,
                max_new_tokens=max_length,
            )
            outputs = outputs[0, inputs["input_ids"].size(1) :]

            if len(outputs) < max_length:
                output_text = tokenizer.decode(outputs, skip_special_tokens=True)
                print(
                    "Text length is less than max_length: ",
                    len(outputs),
                    output_text,
                )
                continue

            outputs = corruption_func(outputs, corruption_fraction)

            # Compute p-value
            if isinstance(wat, UnbiasedWatermarker):
                token_ids = inputs["input_ids"][-wat.config.prefix_length :][0]
                token_ids = torch.cat([token_ids, outputs], dim=0).unsqueeze(0)
                result = wat.p_value(token_ids, model=model)
                all_token_ids.append(token_ids)
            else:
                result = wat.p_value(outputs, n_samples=10000)
                all_token_ids.append(outputs)
            p_values.append(result)
            break

    p_values_tensor = torch.tensor(p_values)
    p_value_median = float(torch.median(p_values_tensor))

    metrics = {
        "p_value_median": p_value_median,
        "p_value_mean": float(p_values_tensor.mean()),
        "p_value_min": float(p_values_tensor.min()),
        "p_value_max": float(p_values_tensor.max()),
        "n_samples": len(p_values),
    }
    tables = {}
    tables["p_values"] = wandb.Table(data=[[p] for p in p_values], columns=["p_value"])

    print(
        f"[{wat_name} | corruption_fraction={corruption_fraction}] median={p_value_median:.4f} ({len(p_values)} samples)"
    )

    if evaluate_scores:
        pos_scores = []
        neg_scores = []

        for token_ids in all_token_ids:
            if isinstance(wat, UnbiasedWatermarker):
                pos_scores.append(wat.test_statistic(model, token_ids))
            else:
                pos_scores.append(wat.test_statistic(token_ids))

        for i in tqdm(range(dataset_size)):
            for j in range(10):
                inputs = data[i].to(device)
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_length,
                )
                outputs = outputs[0, inputs["input_ids"].size(1) :]

                if len(outputs) < max_length:
                    output_text = tokenizer.decode(outputs, skip_special_tokens=True)
                    print(
                        "Text length is less than max_length: ",
                        len(outputs),
                        output_text,
                    )
                    continue

                outputs = corruption_func(outputs, corruption_fraction)

                if isinstance(wat, UnbiasedWatermarker):
                    token_ids = inputs["input_ids"][-wat.config.prefix_length :][0]
                    outputs = torch.cat([token_ids, outputs], dim=0).unsqueeze(0)

                if isinstance(wat, UnbiasedWatermarker):
                    neg_scores.append(wat.test_statistic(model, outputs))
                else:
                    neg_scores.append(wat.test_statistic(outputs))
                break

        print("pos_scores:", pos_scores)
        print("neg_scores:", neg_scores)

        roc_auc = calc_roc_auc_score(pos_scores, neg_scores)
        metrics["roc_auc"] = roc_auc
        print(f"[{wat_name} | corruption_fraction={corruption_fraction}] ROC AUC: {roc_auc:.4f}")

        tpr_at_fpr = calc_tpr_at_fpr(pos_scores, neg_scores, fpr_threshold=0.01)
        metrics["tpr_at_fpr"] = tpr_at_fpr
        print(
            f"[{wat_name} | corruption_fraction={corruption_fraction}] TPR at FPR=0.01: {tpr_at_fpr:.4f}"
        )

        tables["pos_scores"] = wandb.Table(data=[[s] for s in pos_scores], columns=["pos_score"])
        tables["neg_scores"] = wandb.Table(data=[[s] for s in neg_scores], columns=["neg_score"])

    run = init_wandb(
        project=f"WEPA-{experiment}",
        name=f"{model_name}_{wat_name}_len{max_length}",
        config={
            "experiment": experiment,
            "corruption_type": corruption_type,
            "dataset_size": dataset_size,
            "model": model_name,
            "algo": wat_name,
            "max_length": max_length,
            "corruption_fraction": corruption_fraction,
            "device": str(device),
        },
    )

    # Log scalar stats
    wandb.log(metrics)

    # Log tabular data
    for key, value in tables.items():
        wandb.log({key: value})

    run.finish()

In [6]:
from utils import (
    load_default_wepa_watermarker,
    load_default_exp_watermarker,
    load_default_kgw_watermarker,
    load_default_unbiased_watermarker,
)

wats = [
    load_default_wepa_watermarker(
        model.config.vocab_size,
        device,
        degree=1,
    ),
    load_default_wepa_watermarker(
        model.config.vocab_size,
        device,
        degree=2,
    ),
    load_default_exp_watermarker(
        model.config.vocab_size,
        device,
    ),
    load_default_kgw_watermarker(
        model.config.vocab_size,
        device,
    ),
    load_default_unbiased_watermarker(
        model.config.vocab_size,
        device,
    ),
]

for wat in wats:
    for corruption_fraction in corruption_fractions:
        run_experiment(wat, corruption_fraction)

Running experiment: <watermarker.kgw.KGWWatermarker object at 0x7f7750c7a2d0>


  0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_2221852/3241133375.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  token_ids = torch.tensor(token_ids)
  0%|          | 1/200 [00:01<05:36,  1.69s/it]

Text length is less than max_length:  43  of a new Walmart executive. Céline Guede, the fit-minded former executive director of lululemon, has taken the helm at Walmart's fast-growing strength-training division, as we reported last week.


  1%|          | 2/200 [00:03<06:34,  1.99s/it]

Text length is less than max_length:  36  member of campus security who was an acquaintance of McCluskey and knew her roommate Brooke Pratt about McCluskey “He declined to provide further information and never identified himself to […]


  4%|▍         | 8/200 [00:13<05:16,  1.65s/it]


KeyboardInterrupt: 